# 9 WorkFlow Analista Jr

## Version: Semillerio de 25 (Grupo B, Problema #02)
Basado en `z729_final_junior` + semillerio del notebook senior (`z739`) + mejoras de `NEW201_drift_10semillas` y `z620_WorkFlow_06`.

Cambios respecto de z729 junior:
* `semilla_primigenia = 100103`, `experimento = 9300` (carpeta WF9300, no pisa las anteriores).
* **Semillerio de 25** (`PARAM$FT$semillerio <- 25`, semillas = 25 primos al azar entre 100000 y 1000000 elegidos con la semilla primigenia); un modelo final por semilla y prediccion = promedio.
* **Sin Grid Search**: hiperparametros fijos parecidos al cuadro de la clase (`num_iterations = 64`, `num_leaves = 32`, `min_data_in_leaf = 711`, `feature_fraction = 0.20`, `extra_trees = FALSE`, `learning_rate = 0.12`).
* Variables monetarias a `numeric` antes del drift (la deflacion no se trunca si una columna viene entera) y `set.seed` antes del drift (los empates del rank son al azar).
* `mpayroll_sobre_edad` tambien se arma con `mpayroll_rank` si el metodo es rank.
* Corregidos los nombres `drift_dolarblue` / `drift_dolaroficial` del switch (no existian).
* Modelos guardados en `modelos/`: si se corta la corrida, no se reentrenan los que ya estan.
* Kaggle: cortes 1600 a 2600, nombre de archivo con `s25`, y `PARAM$kaggle$enviar` para generar sin subir; un envio fallido no corta el resto.

### Version FE (corrida 9330)
Agrega las variables ganadoras del laboratorio `z729_FE_experimentos_junior` (ganancia local, 10 semillas):
`PARAM$FE$variante` = "V3" (ventana de 3 meses + lags y deltas 3 y 6 de las 60 variables mas importantes de la corrida 9300). Se quitan `numero_de_cliente` y `foto_mes`.
Para correr la otra variante ganadora: cambiar a `"V2"` **y** cambiar `PARAM$experimento` (la carpeta nueva evita pisar la anterior).
Limite de lo medido: el laboratorio entreno con meses hasta 202103 y undersampling del 20 %; aca se entrena con todos los meses hasta 202107 y todos los datos, asi que la mejora real puede ser distinta.
### Version x5 (corrida 9350): 5 corridas independientes de 25 semillas = 125 modelos
Mismo pipeline y mismas variables que la corrida 9330 (V3), pero el semillerio se repite `PARAM$REP$n` veces (minimo 5):
* Las semillas de cada corrida son primos distintos (se sortean todos juntos con la semilla primigenia, sin repetir).
  Las primeras 25 son **las mismas de la corrida 9330**, por eso se pueden reutilizar sus modelos ya entrenados (`PARAM$REP$reusar_modelos`).
* Cada corrida graba su `prediccion_corrida_k.txt`; si se corta, al volver a correr se retoma donde quedo.
* La prediccion final es el **promedio de las 125 probabilidades** (= promedio de las 5 corridas). No se elige "la mejor corrida" mirando Kaggle: eso sobreajusta al leaderboard publico.
* Al final se imprime cuanto cambian las posiciones entre corridas (correlacion y solapamiento del top), que mide el ruido de las semillas.
* Solo se suben a Kaggle los cortes del ensamble final (`PARAM$kaggle$cortes`).

### Version SIN PANDEMIA (corrida 9390)
Igual que la version x5 (V3, 5 corridas x 25 semillas), pero el **modelo final no entrena con los meses 202003 a 202012**.
Motivo: la validacion temporal (`z729_confirmar_sin_pandemia_25`, tres meses, 25 semillas, ensambles de 25 modelos) dio, en ganancia local media: base 78.70 M, sin pandemia 81.45 M (+3.5 %), mezcla de ambas 80.28 M. Sin pandemia mejoro en 202106 (+5.3 M) y 202107 (+4.4 M) y empeoro en 202105 (-1.4 M): ese mes tenia pocos meses posteriores a la pandemia para entrenar; para 202109 hay siete (202101 a 202107).
Las variables (V3) se calculan con TODA la historia; solo cambia en que filas se entrena. No hay modelos reutilizables de la corrida 9330 (otro training): `PARAM$REP$reusar_modelos <- ""`.


### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")


#### Parametros

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 100103

PARAM$experimento <- 9390
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

# --- Feature Engineering ganador del laboratorio (z729_FE_experimentos_junior, 10 semillas) ---
#  "V0": sin variables nuevas | "V1": ventana 3 meses | "V2": lags y deltas 3 y 6 | "V3": V1 + V2
#  Resultado del laboratorio (ganancia local, mejora contra la base):
#     V1 +0.76 M (6/10 semillas, p=0.22) | V2 +3.10 M (10/10, p=0.0059) | V3 +3.61 M (9/10, p=0.0080)
PARAM$FE$variante <- "V3"
PARAM$FE$top_vars <- 60
PARAM$FE$impo <- "/content/buckets/b1/exp/WF9300/impo.txt"    # importancia de la corrida 9300

# Proteccion: si la carpeta del experimento ya existe y tiene archivos, el notebook se detiene (no pisa nada).
PARAM$permitir_carpeta_existente <- FALSE

# --- Repeticion del semillerio ---
PARAM$REP$n <- 5          # cantidad de corridas independientes (cada una con su propio semillerio). Minimo 5.
stopifnot( PARAM$REP$n >= 5 )
# modelos ya entrenados de la corrida 9330 (mismos datos, mismas variables, mismos hiperparametros):
#  si el modelo de una semilla ya existe ahi, se copia en vez de reentrenarlo. Poner "" para entrenar todo de nuevo.
PARAM$REP$reusar_modelos <- ""   # no se reutiliza nada: el training es distinto (sin pandemia)


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
# no pisar corridas anteriores
if( dir.exists(experimento_folder) &&
    length( list.files(experimento_folder, all.files= TRUE, no.. = TRUE) ) > 0 &&
    !PARAM$permitir_carpeta_existente )
  stop( paste0( "La carpeta ", experimento_folder, " ya existe y no esta vacia: NO se reutiliza para no pisar resultados.\n",
    "  Cambie PARAM$experimento por un numero nuevo.\n",
    "  (para retomar una corrida cortada: PARAM$permitir_carpeta_existente <- TRUE)" ) )

dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

In [ ]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [ ]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [ ]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [ ]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [ ]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [ ]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

In [ ]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [ ]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [ ]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [ ]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [ ]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [ ]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [ ]:
# funcion provista por alumnos amantes de la Estadistica Clasica

drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(get(campo), na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

# las variables monetarias pasan a numerico: si alguna columna viene como entera,
#  la deflacion / el dolar no se trunca
dataset[, (campos_monetarios) := lapply(.SD, as.numeric), .SDcols= campos_monetarios]

# los rank con ties.method="random" consumen numeros aleatorios:
#  con la semilla fija el resultado es reproducible
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")

stopifnot( !anyNA( tb_indices$IPC ), nrow(tb_indices) == 33 )

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolar_blue(campos_monetarios),
  "dolar_oficial"  = drift_dolar_oficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

In [ ]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# con los metodos rank, mpayroll ya no existe (queda mpayroll_rank):
#  asi la variable existe en todos los metodos y la comparacion es pareja
if( !("mpayroll_sobre_edad" %in% colnames(dataset)) &&
    atributos_presentes( c("mpayroll_rank", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll_rank / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


#### 9.3.1.5b  FE adicional: variables nuevas para las variables mas importantes
Ganador del laboratorio de variables (10 semillas, ganancia local en 202105 y 202107). Se calculan solo para las `PARAM$FE$top_vars` variables base mas importantes de la corrida 9300.
* **ventana 3 meses** (V1, V3): promedio, minimo y maximo del mes actual y los 2 anteriores.
* **lags y deltas de orden 3 y 6** (V2, V3).

In [ ]:
cols_fe_nuevas <- character(0)

if( PARAM$FE$variante != "V0" )
{
  impo <- fread( PARAM$FE$impo )
  impo[, base := sub("_(lag|delta)[0-9]+$", "", Feature) ]
  tb_base <- impo[, list(gain= sum(Gain)), by= base ][ order(-gain) ]

  vars_top <- tb_base[ base %in% colnames(dataset) &
    !(base %in% c("numero_de_cliente", "foto_mes", "kmes")) &
    paste0(base, "_lag2") %in% colnames(dataset), base ]
  vars_top <- head( vars_top, PARAM$FE$top_vars )
  cat( "variantes de FE:", PARAM$FE$variante, "| variables base:", length(vars_top), "\n" )

  setorder( dataset, numero_de_cliente, foto_mes )

  # ---- ventana de 3 meses (el mes actual y los dos anteriores) ----
  if( PARAM$FE$variante %in% c("V1", "V3") )
  {
    for( v in vars_top )
    {
      m <- cbind( dataset[[v]], dataset[[paste0(v, "_lag1")]], dataset[[paste0(v, "_lag2")]] )
      hay <- rowSums( !is.na(m) ) > 0
      prom <- rowMeans( m, na.rm= TRUE );  prom[ !hay ] <- NA
      set( dataset, j= paste0(v, "_w3mean"), value= prom )
      set( dataset, j= paste0(v, "_w3min"),  value= suppressWarnings( do.call(pmin, c(as.data.frame(m), na.rm= TRUE)) ) )
      set( dataset, j= paste0(v, "_w3max"),  value= suppressWarnings( do.call(pmax, c(as.data.frame(m), na.rm= TRUE)) ) )
    }
    rm(m)
  }

  # ---- lags y deltas de orden 3 y 6 ----
  if( PARAM$FE$variante %in% c("V2", "V3") )
  {
    for( k in c(3, 6) )
    {
      dataset[, paste0(vars_top, "_lag", k) := shift(.SD, k, NA, "lag"), by= numero_de_cliente, .SDcols= vars_top ]
      for( v in vars_top )
        set( dataset, j= paste0(v, "_delta", k), value= dataset[[v]] - dataset[[paste0(v, "_lag", k)]] )
    }
  }
  gc(full= TRUE, verbose= FALSE)
}
ncol(dataset)

Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

# Problema #16 (Grupo B, 10 semillas): sacar numero_de_cliente Y foto_mes juntos gana (p = 0.0059). El laboratorio
#  de variables tambien se midio sin estos dos campos.
campos_buenos <- setdiff( campos_buenos, c("numero_de_cliente", "foto_mes") )
cat( "campos buenos:", length(campos_buenos), "\n" )


####  9.3.2.2. Hiperparametros FIJOS (sin Grid Search)

Se **elimina el Grid Search** (65 minutos) y se usan hiperparametros fijos, parecidos a los del cuadro de la clase
("Parametros inesperados, 100-semillerio"): modelos chicos y muy rapidos, con `feature_fraction` bajo, `min_data_in_leaf` alto y pocas iteraciones;
el poder de prediccion lo da el **semillerio de 25 semillas** (promedio de 25 modelos), no un solo modelo muy ajustado.

| hiperparametro | cuadro de la clase | este notebook |
|---|---|---|
| num_iterations | 23 | **64** (pedido) |
| learning_rate | 0.34 | **0.12** (con 64 iteraciones en vez de 23, se baja el paso para que learning_rate x iteraciones quede parecido: 0.34 x 23 = 7.8 ; 0.12 x 64 = 7.7) |
| num_leaves | 32 | 32 |
| min_data_in_leaf | 711 | 711 |
| feature_fraction | 0.20 | 0.20 |
| extra_trees | FALSE | FALSE |

Sin early stopping: el modelo se entrena exactamente 64 iteraciones.

In [ ]:
# parametros fijos del LightGBM (no hay optimizacion de hiperparametros)
PARAM$lgbm$param_final <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  extra_trees= FALSE,
  num_threads= 4,   # para poder correr en paralelo con la bayesiana (la VM tiene 8 vCPU)

  num_iterations= 64,
  learning_rate= 0.12,
  num_leaves= 32,
  min_data_in_leaf= 711,
  feature_fraction= 0.20
)

PARAM$lgbm$param_final

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002,   # sin 202003 a 202012 (pandemia)
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

##### Final Training Hyperparameters

In [ ]:
# hiperparametros fijos (ver arriba); la semilla se cambia en cada modelo del semillerio
param_final <- copy( PARAM$lgbm$param_final )
param_final

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

##### Training: 5 corridas x 25 semillas
Se entrenan `PARAM$REP$n` corridas de `PARAM$FT$semillerio` modelos cada una (todas con los mismos hiperparametros y sobre TODOS los datos de final_train), cada modelo con una semilla distinta.


In [ ]:
# semillas por corrida (el total es PARAM$REP$n * PARAM$FT$semillerio)
PARAM$FT$semillerio <- 25


In [ ]:
if(!require("primes")) install.packages("primes")
require("primes")

In [ ]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")

# sorteo todas las semillas juntas, sin repetir: las primeras 25 son las de la corrida 9330
semillas_todas <- sample(primos)[ seq(PARAM$FT$semillerio * PARAM$REP$n) ]
stopifnot( !anyDuplicated(semillas_todas) )

# una lista con las semillas de cada corrida
PARAM$REP$semillas <- split( semillas_todas, rep( seq_len(PARAM$REP$n), each= PARAM$FT$semillerio ) )

for( k in seq_len(PARAM$REP$n) )  cat( "corrida", k, ":", PARAM$REP$semillas[[k]], "\n\n" )


In [ ]:
# un modelo final por semilla; si el modelo ya existe en disco, no se reentrena
#  (asi se puede retomar una corrida cortada)
primero <- TRUE

# devuelve "existe", "reusado" o "entrenado"
crear_modelo_final <- function( sem ) {

  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if( file.exists(nombre_arch) ) return( "existe" )

  # modelo ya entrenado en la corrida anterior (9330): se copia
  previo <- file.path( PARAM$REP$reusar_modelos, paste0("modelo_", sem, ".txt") )
  if( nzchar(PARAM$REP$reusar_modelos) && file.exists(previo) )
  {
    file.copy( previo, nombre_arch )
    return( "reusado" )
  }

  param_final$seed <- sem

  set.seed(sem, kind = "L'Ecuyer-CMRG")
  final_model <- lgb.train(
    data= dfinal_train,
    param= param_final,
    verbose= -100
  )

  lgb.save(final_model, nombre_arch) # grabo el modelo

  # grabo la importancia de variables del primer modelo entrenado
  if( primero )
  {
    primero <<- FALSE
    fwrite( as.data.table(lgb.importance(final_model)),
      file= "impo.txt",
      sep= "\t"
    )
  }

  rm(final_model)
  gc(full = TRUE, verbose=FALSE)
  return( "entrenado" )
}


In [ ]:
gc(full = TRUE, verbose=FALSE)
dir.create("modelos", showWarnings =FALSE)

# datos del futuro: cada modelo se aplica apenas se crea
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])

tb_corridas <- dfuture[, list(numero_de_cliente)]

# --- cuantos modelos hay que entrenar de verdad (los que ya existen o se reusan no cuentan)
todas <- unlist( PARAM$REP$semillas )
ya_estan <- file.exists( paste0("./modelos/modelo_", todas, ".txt") ) |
  ( nzchar(PARAM$REP$reusar_modelos) &
    file.exists( file.path(PARAM$REP$reusar_modelos, paste0("modelo_", todas, ".txt")) ) )
pendientes <- sum( !ya_estan )
cat( "modelos totales:", length(todas), "| a entrenar:", pendientes, "| ya existentes o reusables:", sum(ya_estan), "\n" )

t0 <- Sys.time()
entrenados <- 0
hechos <- 0

for( k in seq_len(PARAM$REP$n) )
{
  archivo_pred <- sprintf( "prediccion_corrida_%d.txt", k )

  # si la corrida ya esta terminada (corrida cortada), la leo y sigo
  if( file.exists(archivo_pred) )
  {
    tb_k <- fread( archivo_pred )
    stopifnot( nrow(tb_k) == nrow(dfuture), all( tb_k$numero_de_cliente == dfuture$numero_de_cliente ) )
    tb_corridas[, paste0("prob_c", k) := tb_k$prob ]
    hechos <- hechos + length( PARAM$REP$semillas[[k]] )
    cat( "corrida", k, "ya estaba terminada\n" )
    next
  }

  semillas_k <- PARAM$REP$semillas[[k]]
  prob_k <- numeric( nrow(dfuture) )

  for( i in seq_along(semillas_k) )
  {
    sem <- semillas_k[i]
    estado <- crear_modelo_final( sem )
    if( estado == "entrenado" ) entrenados <- entrenados + 1
    hechos <- hechos + 1

    final_model <- lgb.load( paste0( "./modelos/modelo_", sem, ".txt") )
    prob_k <- prob_k + predict( final_model, datos_matrix )
    rm(final_model)

    # --- update de progreso (se imprime y se graba en progreso.txt)
    min_pasados <- as.numeric( difftime(Sys.time(), t0, units= "mins") )
    min_por_modelo <- if( entrenados > 0 ) min_pasados / entrenados else 0
    faltan <- max( pendientes - entrenados, 0 )
    eta <- Sys.time() + min_por_modelo * faltan * 60

    linea <- sprintf("%s  corrida %d/%d  modelo %d/%d (%s)  total %d/%d (%.0f%%)  %.1f min/modelo  termina aprox %s",
      format(Sys.time(), "%H:%M:%S"), k, PARAM$REP$n, i, length(semillas_k), estado,
      hechos, length(todas), 100 * hechos / length(todas), min_por_modelo, format(eta, "%a %d %H:%M"))
    cat( linea, "\n" )
    cat( linea, "\n", file= "progreso.txt", append= TRUE )
  }
  gc(full = TRUE, verbose=FALSE)

  prob_k <- prob_k / length(semillas_k)
  fwrite( data.table( numero_de_cliente= dfuture$numero_de_cliente, prob= prob_k ),
    file= archivo_pred, sep= "\t" )
  tb_corridas[, paste0("prob_c", k) := prob_k ]
  cat( "== corrida", k, "terminada ==\n" )
}

rm( datos_matrix )
cat( "modelos en disco:", length(list.files("modelos")), "de", length(todas), "\n" )


#### Scoring

Los modelos se aplican a los datos del futuro dentro del bloque de entrenamiento (arriba). Aca se juntan las corridas en una sola prediccion.

In [ ]:
# (dfuture ya se definio en el bloque de entrenamiento)


In [ ]:
# --- cuanto cambian las posiciones entre corridas (ruido de las semillas) ---
cols_c <- paste0( "prob_c", seq_len(PARAM$REP$n) )

cat( "\ncorrelacion de posiciones (Spearman) entre corridas:\n" )
print( round( cor( tb_corridas[, cols_c, with= FALSE], method= "spearman" ), 4 ) )

corte_ref <- 1900
tops <- lapply( cols_c, function(cn) tb_corridas[ order(-get(cn)) ][ seq_len(corte_ref), numero_de_cliente ] )
solap <- sapply( seq_along(tops), function(a) sapply( seq_along(tops), function(b) length( intersect(tops[[a]], tops[[b]]) ) / corte_ref ) )
cat( "\nsolapamiento de los", corte_ref, "primeros entre corridas (1 = identicos):\n" )
print( round( solap, 3 ) )
cat( "solapamiento promedio entre pares:", round( mean( solap[ upper.tri(solap) ] ), 3 ), "\n" )

# --- ensamble final = promedio de las probabilidades de todas las corridas (125 modelos) ---
tb_prediccion <- copy( tb_corridas )
tb_prediccion[, prob := rowMeans( .SD ), .SDcols= cols_c ]

PARAM$FT$semillas <- todas   # para que el nombre y el mensaje de Kaggle cuenten los 125 modelos


##### Tabla Prediccion

In [ ]:
# grabo las probabilidades del ensamble
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

tb_prediccion[, list(numero_de_cliente, prob)]

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"
PARAM$kaggle$cortes <- seq(1900, 2300, by = 100)   # zona del techo de la curva local (2150-2200) y del nivel que rindio bien en Kaggle

# TRUE = sube a Kaggle; FALSE = solo genera los archivos
PARAM$kaggle$enviar <- FALSE   # en la corrida original estaba en TRUE (subia los cortes a Kaggle); aca solo genera los archivos

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_s", length(PARAM$FT$semillas), "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  if( !PARAM$kaggle$enviar ) next

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  sin2020  modelos=", length(PARAM$FT$semillas), "  corridas=", PARAM$REP$n,
    "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  # si un envio falla, se informa y se sigue con el siguiente
  salida <- tryCatch(
    system(linea, intern=TRUE), # el submit a Kaggle
    error= function(e) conditionMessage(e),
    warning= function(w) conditionMessage(w)
  )
  Sys.sleep(30)
  cat(envios, ":", salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")